# Word Embeddings : le modèle Word2Vec

## Imports

In [58]:
import sys
import os

from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec

import nltk
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

## Chargement et traitement des phrases du corpus

### Création d'un objet qui *streame* les lignes d'un fichier pour économiser de la RAM

In [59]:
class MySentences(object):
    """Tokenize and Lemmatize sentences"""
    def __init__(self, filename):
        self.filename = filename

    def __iter__(self):
        for line in open(self.filename, encoding='utf-8', errors="backslashreplace"):
            yield [unidecode(w.lower()) for w in wordpunct_tokenize(line)]

In [60]:
infile = f"../../data/sents1.txt"
sentences = MySentences(infile)

### Détection des bigrams

Article intéressant sur le sujet : https://towardsdatascience.com/word2vec-for-phrases-learning-embeddings-for-more-than-one-word-727b6cf723cf

In [61]:
bigram_phrases = Phrases(sentences)

L'object `phrases` peut être vu comme un large dictionnaire d'expressions multi-mots associées à un score, le *PMI-like scoring*. Ce dictionnaire est construit par un apprentissage sur base d'exemples.
Voir les références ci-dessous :
- https://arxiv.org/abs/1310.4546
- https://en.wikipedia.org/wiki/Pointwise_mutual_information

In [62]:
type(bigram_phrases.vocab)

dict

Il contient de nombreuses clés qui sont autant de termes observés dans le corpus

In [63]:
len(bigram_phrases.vocab.keys())

4310398

Prenons une clé au hasard :

In [64]:
key_ = list(bigram_phrases.vocab.keys())[144]
print(key_)

1q


Le dictionnaire indique le score de cette coocurrence :

In [65]:
bigram_phrases.vocab[key_]

42

Lorsque l'instance de `Phrases` a été entraînée, elle peut concaténer les bigrams dans les phrases lorsque c'est pertinent.

### Conversion des `Phrases` en objet `Phraser`

`Phraser` est un alias pour `gensim.models.phrases.FrozenPhrases`, voir ici https://radimrehurek.com/gensim/models/phrases.html.

Le `Phraser` est une version *light* du `Phrases`, plus optimale pour transformer les phrases en concaténant les bigrams.

In [66]:
bigram_phraser = Phraser(phrases_model=bigram_phrases)

Le `Phraser` est un objet qui convertit certains unigrams d'une liste en bigrams lorsqu'ils ont été identifiés comme pertinents.

### Extraction des trigrams

Nous répétons l'opération en envoyant cette fois la liste de bigrams afin d'extraire les trigrams.

In [67]:
trigram_phrases = Phrases(bigram_phraser[sentences])

In [68]:
trigram_phraser = Phraser(phrases_model=trigram_phrases)

### Création d'un corpus d'unigrams, bigrams, trigrams

In [69]:
corpus = list(trigram_phraser[bigram_phraser[sentences]])

In [70]:
print(corpus[:100])

[['mi', 'imnri', 'r', 'i', '<<', 'i', 'i', 'hmu', "'", 'i', '/', 'tx', "-'", 'l', ':', 'marche', 'tenu', 'hors', 'villa', ',', 'la', '9', '.'], ['--', 'u', 'a', 'ete', 'vaain', 'si', 'teicj', '>>', 'm', 'races', 'indigenes', 'de', 'fr', '.'], ['31', '<)', 'a', '5s', "'", 'k', '131', 'de', '.'], ['rasa', 'iichakdui', "'", 'te', ',', 'do', '(', 'r', '.', '3s0', 'h', '710', '.', 'taureaux', 'iallsenas', ',>', 'ia', 'u', '\\', '--', 'a', '--', ';', '0ii', '.'], ['hollandais', ',', 'dufr', '.'], ['0', '.'], ['--', 'a', '9', '.--', 'la', 'idto', '-', 'vachei', 'laitieres', ':', 'bn', 'vante', '1q', '.'], ['vendues', '3', '\\', 'au', 'prix', 'la', '410', 'a', '*', '<<', 'i', 'h', '\\;', 'genisses', ',', 'kl', '.'], ["'.", '9', '.'], ['i', 'l', '.', '2', 'i', '.', 'id', '.'], ['da', '370', 'i', '6lutr', '.'], ['marche', 'a', '<', 'u', 'porcs', '.'], ['--', 'categorie', 'de', 'lt', 'ilashtya', ':', "'", '237', 'on', 'vente', ';', 'vendus', '1', 'm', '.', 'do', "'", '2', 'i', '.--', 'a', ';:,', 

## Entrainement d'un modèle Word2Vec sur ce corpus

In [72]:
data_path = "../../data/txt/"
temp_path = os.path.join(os.path.dirname(data_path.rstrip("/")), "tmp")
os.makedirs(temp_path, exist_ok=True)

In [73]:
configurations = [
    {"window": 3, "min_count": 5},
    {"window": 5, "min_count": 5},
    {"window": 5, "min_count": 10},
]

modeles = {}
for cfg in configurations:
    nom = f"w{cfg['window']}_m{cfg['min_count']}"
    modele = Word2Vec(
        corpus,
        vector_size=32,
        window=cfg["window"],
        min_count=cfg["min_count"],
        workers=4,
        epochs=5,
    )
    modele.save(os.path.join(temp_path, f"{nom}.model"))
    modeles[nom] = modele
    print(nom, "→", len(modele.wv), "mots dans le vocabulaire")

w3_m5 → 105500 mots dans le vocabulaire
w5_m5 → 105500 mots dans le vocabulaire
w5_m10 → 64883 mots dans le vocabulaire


#### Remarque

Vous voyez ici que l'entrainement du modèle est parallélisé (sur 4 workers).

Lors qu'on parallélise l'entrainement du modèle, 4 modèles "séparés" sont entrainés sur environ un quart des phrases.

Ensuite, les résultats sont agrégés pour ne plus faire qu'un seul modèle.

On ne peut prédire quel worker aura quelle phrase, car il y a des aléas lors de la parallélisation (p. ex. un worker qui serait plus lent, etc.).

Du coup, les valeurs peuvent varier légèrement d'un entrainement à l'autre.

Mais, globalement, les résultats restent cohérents.

### Sauver le modèle dans un fichier

In [74]:
for nom, modele in modeles.items():
    if "ministre" in modele.wv:
        voisins = [mot for mot, score in modele.wv.most_similar("ministre", topn=5)]
        print(f"{nom} : {', '.join(voisins)}")
    else:
        print(f"{nom} : « ministre » absent du vocabulaire")

w3_m5 : secretaire, chevalier, directeur, professeur, membre
w5_m5 : secretaire, president, vice_-_president, en_remplacement, adjoint
w5_m10 : secretaire, membre, president, chevalier, conseiller


## Explorer le modèle

### choix du modèle w5_m5 (window=5, min_count=5) apres comparaison des voisins de « ministre »

In [75]:
MEILLEUR = "w5_m5"
model = Word2Vec.load(os.path.join(temp_path, f"{MEILLEUR}.model"))

Cellule similarity

In [76]:
for a, b in [("ministre", "roi"), ("paris", "londres"), ("belgique", "france")]:
    if a in model.wv and b in model.wv:
        print(f"similarity({a}, {b}) = {model.wv.similarity(a, b):.3f}")
    else:
        print(f"{a} ou {b} absent du vocabulaire")

similarity(ministre, roi) = 0.557
similarity(paris, londres) = 0.884
similarity(belgique, france) = 0.704


Cellule most_similar

In [78]:
for mot in ["ministre", "paris", "football"]:
    if mot in model.wv:
        print(mot, "-", [m for m, s in model.wv.most_similar(mot, topn=5)])
    else:
        print(mot, "absent du vocabulaire")

ministre - ['secretaire', 'president', 'vice_-_president', 'en_remplacement', 'adjoint']
paris - ['londres', 'nice', 'marseille', 'berlin', 'new_-_york']
football - ['challenge', 'match', 'cyclisme', 'tournoi', 'championnat']
